In [14]:
import pandas as pd
import torch
import numpy as np
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

## БЕРЕМ ПРАВИЛЬНЫЕ ДАННЫЕ

In [2]:
data_final = pd.read_csv('final_df.csv')
data_final

,doc_id,chunk_id,is_macro,chunk_text
0,0,0,True,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,2,0,True,"""С 5 февраля ЕС вводит эмбарго на российские п..."
2,2,1,True,Госбюджет РФ рискует недополучить еще больше н...
3,4,2,False,"Катализатора для реализации нашей идеи нет, по..."
4,5,0,True,"""Окно закрывается. С начала СВО физлица вывели..."
...,...,...,...,...
2304,1498,0,False,"""Вероятность перемирия России и Украины до 19 ..."
2305,1499,0,False,"""Потрясения в банковской сфере США вновь оказы..."
2306,1499,1,False,"Между тем, обеспокоенность по поводу потолка а..."
2307,1499,2,False,"Среди лидеров падения: PacWest Bancorp (-26,5%..."


In [3]:
data_final_train  = data_final[data_final['doc_id'] < 1200]
data_final_val = data_final[data_final['doc_id'] >= 1200]

In [4]:
data_final_train.shape, data_final_val.shape

((1877, 4), (432, 4))

In [5]:
## очень плохо с дизбалансом, выбросим последние 300 данных
data_final_train['is_macro'].mean(), data_final_val['is_macro'].mean()

(np.float64(0.38625466169419287), np.float64(0.2037037037037037))

In [6]:
data_train_new = data_final_train[data_final_train['doc_id'] < 1000]
data_val_new = data_final_train[data_final_train['doc_id'] >= 1000]

In [7]:
data_train_new.shape, data_val_new.shape

((1540, 4), (337, 4))

In [8]:
## остальные данные сделаем тестом(в реальности будет дизбаланс)
data_test_new = data_final_val

In [9]:
## вот сейчас все ок
data_train_new['is_macro'].mean(), data_val_new['is_macro'].mean()

(np.float64(0.37922077922077924), np.float64(0.41839762611275966))

In [10]:
# КОНФИГУРАЦИЯ
MODEL_NAME = "ai-forever/ruBert-base"
DATA_PATH = "your_data.csv"
OUTPUT_DIR = "./macro_bert_model"
MAX_LEN = 300

# ПОДГОТОВКА ДАННЫХ
class news_dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_texts, val_texts, train_labels, val_labels = data_train_new['chunk_text'],  data_val_new['chunk_text'], data_train_new['is_macro'].astype('int'), data_val_new['is_macro'].astype('int')
## ОБРАБОТКА
train_texts = [str(t) if t is not None else "" for t in train_texts]
val_texts = [str(t) if t is not None else "" for t in val_texts]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LEN)

train_dataset = news_dataset(train_encodings, train_labels)
val_dataset = news_dataset(val_encodings, val_labels)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [15]:

train_texts, val_texts, train_labels, val_labels = data_train_new['chunk_text'],  data_val_new['chunk_text'], data_train_new['is_macro'].astype('int'), data_val_new['is_macro'].astype('int')
## НА ВСЯКИЙ СЛУЧАЙ
train_texts = [str(t) if t is not None else "" for t in train_texts]
val_texts = [str(t) if t is not None else "" for t in val_texts]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LEN)

train_dataset = news_dataset(train_encodings, train_labels)
val_dataset = news_dataset(val_encodings, val_labels)

# РАСЧЕТ ВЕСОВ КЛАССОВ
class_weights = len(train_labels) / (2 * np.bincount(train_labels))
class_weights = torch.tensor(class_weights, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")

# КАСТОМНЫЙ ТРЕЙНЕР - ОЧЕНЬ ПРОСТОЙ В ИСПОЛЬЗОВАНИИ(TRAINER)
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss() ## ТЕСТИЛ С ВЕСАМИ И БЕЗ, БЕЗ ВЕСОВ ПОЛУЧАЛОСЬ ЛУЧШЕ
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# МЕТРИКИ
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# АРГУМЕНТЫ ОБУЧЕНИЯ
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,               # если больше, то БУДЕТ ПЕРЕОБУЧЕНИЕ
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,               # ЕЕСЛИ НЕМНОГО МЕНЯТЬ, ТО ВСЕ ПАДАЕТ
    warmup_steps=100,                 # Если 1000 то уже не обучается
    weight_decay=0.05,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,      # Вернет лучшую модель
    fp16=torch.cuda.is_available(),   # Ускорение на GPU
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
)

print("Начинаю обучение...")
trainer.train()

# Сохраняем результат
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Модель сохранена в {OUTPUT_DIR}")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

Начинаю обучение...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.634915,0.199285,0.928783,0.920530
2,0.187004,0.164813,0.949555,0.941980
3,0.084439,0.180355,0.952522,0.944444


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в ./macro_bert_model


In [16]:
!zip -r macro_bert_model.zip ./macro_bert_model

  adding: macro_bert_model/ (stored 0%)
  adding: macro_bert_model/model.safetensors (deflated 7%)
  adding: macro_bert_model/checkpoint-97/ (stored 0%)
  adding: macro_bert_model/checkpoint-97/trainer_state.json (deflated 57%)
  adding: macro_bert_model/checkpoint-97/scheduler.pt (deflated 61%)
  adding: macro_bert_model/checkpoint-97/optimizer.pt (deflated 49%)
  adding: macro_bert_model/checkpoint-97/training_args.bin (deflated 53%)
  adding: macro_bert_model/checkpoint-97/model.safetensors (deflated 7%)
  adding: macro_bert_model/checkpoint-97/tokenizer.json (deflated 73%)
  adding: macro_bert_model/checkpoint-97/config.json (deflated 56%)
  adding: macro_bert_model/checkpoint-97/tokenizer_config.json (deflated 42%)
  adding: macro_bert_model/checkpoint-97/scaler.pt (deflated 64%)
  adding: macro_bert_model/checkpoint-97/rng_state.pth (deflated 26%)
  adding: macro_bert_model/checkpoint-194/ (stored 0%)
  adding: macro_bert_model/checkpoint-194/trainer_state.json (deflated 63%)
  a

### ЗАГРУЗИМ В ОБЛАКО

In [18]:
!curl -F "file=@./macro_bert_model.zip" https://store1.gofile.io/uploadFile

{"data":{"createTime":1776949052,"downloadPage":"https://gofile.io/d/nBLHBY","guestToken":"PDatzl3On0whvQYgiSoKuOBxWl1HRzGG","id":"e94a385c-607b-4f74-ace5-dde1dada113b","md5":"dbcb89fe522e91a2d685b70fb75eb6b5","mimetype":"application/zip","modTime":1776949052,"name":"macro_bert_model.zip","parentFolder":"9f568dfd-f699-4dec-a29b-1681fe781e79","parentFolderCode":"nBLHBY","servers":["store1"],"size":4855786623,"type":"file"},"status":"ok"}
